# Experimento completo em GPU (Kaggle / Colab)

Segmentação do Gray-Scott conforme o `PROTOCOLO.MD` da turma: **U-Net, SegFormer e Mask2Former**, 20 épocas (melhor época pela validação), baseline + P1–P10, baseline + Q1–Q5.

**Kaggle:** *Session options* → Accelerator **GPU T4 x2**, Internet **On** → **Save Version → Save & Run All (Commit)**.

- As 2 GPUs dividem uma fila de 33 treinos (cada uma pega o próximo livre; Mask2Former primeiro).
- **Se estourar as 12 h:** abra uma nova versão, em *Add Input* anexe a saída da versão anterior (*Notebook Output*) e rode de novo. Os treinos prontos são reaproveitados e só o que falta é treinado.
- No fim, baixe `resultados.zip` na aba **Output** e descompacte na raiz do projeto local.

In [ ]:
import os, subprocess
REPO = 'https://github.com/JoaoLago10/segmentacao-gray-scott.git'
root = '/kaggle/working' if os.path.isdir('/kaggle/working') else os.path.expanduser('~')
dest = os.path.join(root, 'projeto')          # caminho absoluto: rodar esta célula de novo não duplica a pasta
if os.path.isdir(os.path.join(dest, '.git')):
    subprocess.run(['git', '-C', dest, 'pull', '-q'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', REPO, dest], check=True)
os.chdir(dest)
print(os.getcwd())
!git log --oneline -1
!nvidia-smi -L

In [ ]:
!pip install -q h5py fsspec aiohttp PyWavelets scikit-image transformers
import torch, transformers; print(torch.__version__, transformers.__version__, torch.cuda.device_count(), 'GPU(s)')

In [ ]:
# Retomada: se a saída de uma versão anterior foi anexada como input, reaproveita treinos e probabilidades.
import glob, shutil, os
prev = [p for p in glob.glob('/kaggle/input/**/results/exp/runs', recursive=True)]
if prev:
    src = os.path.dirname(prev[0])
    print('reaproveitando', src)
    shutil.copytree(src, 'results/exp', dirs_exist_ok=True)
    print(len(os.listdir('results/exp/runs')), 'treinos já prontos')
else:
    print('execução nova (nenhuma saída anterior anexada)')

In [ ]:
import os, subprocess, time
# ~2,8 GB por leitura parcial do Hugging Face. Falhas temporárias (limite de taxa) são repetidas;
# o que já foi baixado fica em cache por regime, então uma nova tentativa continua de onde parou.
# Opcional: adicione um token do Hugging Face em Add-ons -> Secrets com o nome HF_TOKEN (limites maiores).
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('usando HF_TOKEN')
except Exception:
    print('sem HF_TOKEN (requisições anônimas)')
for tentativa in range(1, 4):
    r = subprocess.run('python scripts/04_preparar_dados.py > log_dados.txt 2>&1', shell=True)
    print(f'tentativa {tentativa}: código {r.returncode}')
    print(''.join(open('log_dados.txt').readlines()[-3:]))
    if r.returncode == 0:
        break
    time.sleep(120)
else:
    raise RuntimeError('download falhou 3 vezes; veja log_dados.txt')

In [ ]:
# Mede a velocidade real nesta GPU e estima o tempo total por arquitetura (~3 min).
!python scripts/05_treinar.py --bench --amp

In [ ]:
# Treino: fila compartilhada entre as GPUs (cada uma pega o próximo treino livre; Mask2Former primeiro).
import subprocess, time, os, shutil
shutil.rmtree('results/exp/claims', ignore_errors=True)   # reservas de uma sessão anterior
ngpu = max(torch.cuda.device_count(), 1)
procs = []
for gpu in range(ngpu):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    log = open(f'log_gpu{gpu}.txt', 'a')
    procs.append(subprocess.Popen('python scripts/05_treinar.py --arch all --pre all --amp --fila',
                                  shell=True, env=env, stdout=log, stderr=subprocess.STDOUT))
    time.sleep(60)   # evita que as duas baixem os pesos do Hugging Face ao mesmo tempo
t0 = time.time()
while any(p.poll() is None for p in procs):
    time.sleep(600)
    done = len(os.listdir('results/exp/runs')) if os.path.isdir('results/exp/runs') else 0
    last = [open(f'log_gpu{g}.txt').read().splitlines()[-1:] for g in range(ngpu)]
    print(f'[{(time.time() - t0) / 3600:.1f} h] {done}/33 treinos prontos |', ' | '.join(l[0] if l else '' for l in last), flush=True)
for g in range(ngpu):
    print(f'--- GPU {g} (fim do log)'); print('\n'.join(open(f'log_gpu{g}.txt').read().splitlines()[-4:]))
print('códigos de saída:', [p.returncode for p in procs])

In [ ]:
!MPLCONFIGDIR=.cache/mpl python scripts/06_avaliar.py --workers 4
!MPLCONFIGDIR=.cache/mpl python scripts/07_figuras.py
!python scripts/08_tabelas_latex.py

In [ ]:
!cp log_gpu*.txt results/exp/ 2>/dev/null; zip -qr resultados.zip results/exp docs/artigo/figs/exp docs/artigo/tabelas/exp -x 'results/exp/probs/*' 'results/exp/checkpoints/*'
!ls -lh resultados.zip
!rm -rf data/processed   # os dados são baixados de novo se precisar; probabilidades e checkpoints ficam para retomada